In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## X bar chart ARL0 estimation

In [4]:
def simulate_ARL0_xbar(A, n, mu0=0, sigma0=1, n_sim=10000, max_run=100000):
  # max_run：每次模擬最多允許抽取的 subgroup 數，避免在極端情況下長時間沒有 signal 造成無限迴圈
  # 這個函數的目的是模擬製程在正常狀態下，X-bar chart 平均需要連續監控幾組 subgroup 才會第一次發出 signal，並以此估計 ARL0。

  # Step 1：建立一個 list，用來儲存每一次模擬得到的 run length
  # run length 表示從開始監控到第一次發出 signal，總共抽了幾組 subgroup
  # 儲存每一次模擬得到的 run length
  run_lengths = []

  # Step 2：建立 x-bar chart 的管制界限
  # A 建議代入表格的數字，例如 n = 5 時，A = 1.342；用 3 / sqrt(n) 也可以，但跟下面結果會有些微差異
  center_line = mu0
  UCL = center_line + A * sigma0
  LCL = center_line - A * sigma0

  # Step 3：重複進行 n_sim 次模擬
  for _ in range(n_sim): # 填空 5

    # Step 4：初始化本次模擬的 run length 與 signal 狀態
    run_length = 0  # 填空 6
    signal = False  # 填空 7

    # Step 5：持續抽 subgroup，直到第一次超出控制界限，或達到 max_run 為止
    while (not signal) and (run_length < max_run):
      # Step 6：每抽一組 subgroup，run length 增加 1
      run_length += 1  # 填空 8

      # Step 7：產生一組大小為 n 的 subgroup
      subgroup = np.random.normal(loc=mu0, scale=sigma0, size=n)

      # Step 8：計算 subgroup mean，也就是 x-bar
      xbar = np.mean(subgroup)   # 填空 10

      # Step 9：判斷 x-bar 是否超出控制界限
      if (xbar > UCL) or (xbar < LCL): # 填空 11
        signal = True  # 填空 12

    # Step 10：記錄本次模擬得到的 run length
    run_lengths.append(run_length) # 填空 13

  # Step 11：計算所有 run length 的平均值，作為 ARL0 的估計值
  result = {
      "ARL0": np.mean(run_lengths)
  }

  return result


In [5]:
np.random.seed(123)
# x-bar chart 不同 subgroup size 的 A 值
# 每一組設定包含 subgroup size n，以及對應的 X-bar chart 控制界限係數 A
settings = [
    {"n": 2, "A": 2.121},
    {"n": 3, "A": 1.732},
    {"n": 4, "A": 1.500},
    {"n": 5, "A": 1.342},
    {"n": 6, "A": 1.225}
]

xbar_results = []

for setting in settings:
    n = setting["n"]
    A = setting["A"]

    result = simulate_ARL0_xbar(
        A=A,
        n=n
    )

    xbar_results.append({
        "n": n,
        "A": A,
        "ARL0": result["ARL0"]
    })

# 轉成 DataFrame
xbar_result_df = pd.DataFrame(xbar_results)
print(xbar_result_df)

   n      A      ARL0
0  2  2.121  366.3244
1  3  1.732  369.4360
2  4  1.500  374.0375
3  5  1.342  370.1608
4  6  1.225  371.3424


## R chart ARL0 estimation

Assume that the observations in each subgroup are independently generated from an in-control normal process:

$$
X_{ij} \sim N(\mu_0, \sigma_0^2), 
\quad i = 1,2,\ldots, 
\quad j = 1,2,\ldots,n.
$$

where $X_{ij}$ denotes the $j$-th observation in the $i$-th subgroup.

The standardized variable is defined as

$$
Z_{ij} = \frac{X_{ij} - \mu_0}{\sigma_0}.
$$

Therefore,

$$
X_{ij} = \mu_0 + \sigma_0 Z_{ij},
$$

where

$$
Z_{ij} \sim N(0,1).
$$

For the $i$-th subgroup, the range statistic used in the R chart is defined as

$$
R_i = 
\max_{1 \leq j \leq n}(X_{ij}) 
-
\min_{1 \leq j \leq n}(X_{ij}).
$$

Substituting $X_{ij} = \mu_0 + \sigma_0 Z_{ij}$, we obtain

$$
\begin{aligned}
R_i 
&= \max_{1 \leq j \leq n}(\mu_0 + \sigma_0 Z_{ij})
 - \min_{1 \leq j \leq n}(\mu_0 + \sigma_0 Z_{ij}) \\
&= \left[\mu_0 + \sigma_0 \max_{1 \leq j \leq n}(Z_{ij})\right]
 - \left[\mu_0 + \sigma_0 \min_{1 \leq j \leq n}(Z_{ij})\right] \\
&= \sigma_0 \left[
\max_{1 \leq j \leq n}(Z_{ij}) 
-
\min_{1 \leq j \leq n}(Z_{ij})
\right].
\end{aligned}
$$

Let

$$
W_i =
\max_{1 \leq j \leq n}(Z_{ij}) 
-
\min_{1 \leq j \leq n}(Z_{ij}),
$$

where $W_i$ denotes the range of the standardized observations in the $i$-th subgroup. Then,

$$
R_i = \sigma_0 W_i.
$$

Taking expectation on both sides gives

$$
\begin{aligned}
E(R_i) 
&= E(\sigma_0 W_i) \\
&= \sigma_0 E(W_i).
\end{aligned}
$$

The control chart constant $d_2$ is defined as

$$
d_2 = E(W_i).
$$

Thus,

$$
E(R_i) = d_2 \sigma_0.
$$

Similarly, the standard deviation of $R_i$ is

$$
\begin{aligned}
SD(R_i)
&= SD(\sigma_0 W_i) \\
&= \sigma_0 SD(W_i).
\end{aligned}
$$

The control chart constant $d_3$ is defined as

$$
d_3 = SD(W_i).
$$

Therefore,

$$
SD(R_i) = d_3 \sigma_0.
$$

Hence, the center line and control limits of the R chart are

$$
CL = d_2 \sigma_0,
$$

$$
UCL = d_2 \sigma_0 + 3d_3 \sigma_0
= (d_2 + 3d_3)\sigma_0,
$$

$$
LCL = d_2 \sigma_0 - 3d_3 \sigma_0
= (d_2 - 3d_3)\sigma_0.
$$

Since the range statistic cannot be negative, the lower control limit is usually written as

$$
LCL = \max\left\{(d_2 - 3d_3)\sigma_0, 0\right\}.
$$

In [1]:
def simulate_ARL0_rchart(d2, d3,  n, mu0=0, sigma0=1, n_sim=10000, max_run=100000):
  # max_run：每次模擬最多允許抽取的 subgroup 數，避免在極端情況下長時間沒有 signal 造成無限迴圈
  # 這個函數的目的是模擬製程在正常狀態下，R chart 平均需要連續監控幾組 subgroup 才會第一次發出 signal，並以此估計 ARL0。

  # Step 1：建立一個 list，用來儲存每一次模擬得到的 run length
  # run length 表示從開始監控到第一次發出 signal，總共抽了幾組 subgroup
  run_lengths = []

  # Step 2：建立 R chart 的管制界限
  center_line = d2 * sigma0
  UCL = center_line + 3 * d3 * sigma0
  LCL = max(center_line - 3 * d3 * sigma0, 0)

  # Step 3：重複進行 n_sim 次模擬
  for _ in range(n_sim): 
    # Step 4：初始化本次模擬的 run length 與 signal 狀態
    run_length = 0  
    signal = False  

    # Step 5：持續抽 subgroup，直到第一次超出控制界限，或達到 max_run 為止
    while (not signal) and (run_length < max_run):
      # Step 6：每抽一組 subgroup，run length 增加 1
      run_length += 1 

      # Step 7：產生一組大小為 n 的 subgroup
      subgroup = np.random.normal(loc=mu0, scale=sigma0, size=n)

      # Step 8：計算 subgroup range，也就是 R chart 監控的統計量
      R = np.max(subgroup) - np.min(subgroup)

      # Step 9：判斷 R 是否超出控制界限
      if (R > UCL) or (R < LCL):
        signal = True 

    # Step 10：記錄本次模擬得到的 run length
    run_lengths.append(run_length)

  # Step 11：計算所有 run length 的平均值，作為 ARL0 的估計值
  result = {
      "ARL0": np.mean(run_lengths)
  }

  return result


In [3]:
np.random.seed(123)

# R chart 不同 subgroup size 的 d2 和 d3 值
# 每一組設定包含 subgroup size n，以及對應的 R chart 常數 d2 和 d3
settings = [
    {"n": 2, "d2": 1.128, "d3": 0.853},
    {"n": 3, "d2": 1.693, "d3": 0.888},
    {"n": 4, "d2": 2.059, "d3": 0.880},
    {"n": 5, "d2": 2.326, "d3": 0.864},
    {"n": 6, "d2": 2.534, "d3": 0.848}
]

rchart_results = []

for setting in settings:
    n = setting["n"]
    d2 = setting["d2"]
    d3 = setting["d3"]

    result = simulate_ARL0_rchart(
        d2=d2,
        d3=d3,
        n=n
    )

    rchart_results.append({
        "n": n,
        "d2": d2,
        "d3": d3,
        "ARL0": result["ARL0"]
    })

# 轉成 DataFrame
rchart_result_df = pd.DataFrame(rchart_results)

print(rchart_result_df)

   n     d2     d3      ARL0
0  2  1.128  0.853  108.3121
1  3  1.693  0.888  170.9558
2  4  2.059  0.880  204.0438
3  5  2.326  0.864  220.7960
4  6  2.534  0.848  227.0254


## S chart ARL0 estimation

Assume that the observations in each subgroup are independently generated from an in-control normal process:

$$
X_{ij} \sim N(\mu_0, \sigma_0^2),
\quad i = 1,2,\ldots,
\quad j = 1,2,\ldots,n.
$$

where $X_{ij}$ denotes the $j$-th observation in the $i$-th subgroup.

The statistic monitored by the S chart is the sample standard deviation of each subgroup. For the $i$-th subgroup, it is defined as

$$
S_i = \sqrt{\frac{1}{n-1}\sum_{j=1}^{n}(X_{ij}-\bar{X}_i)^2},
$$

where

$$
\bar{X}_i = \frac{1}{n}\sum_{j=1}^{n}X_{ij}.
$$

For normally distributed observations, the sample variance satisfies

$$
\frac{(n-1)S_i^2}{\sigma_0^2} \sim \chi^2_{n-1}.
$$

Although $S_i^2$ is an unbiased estimator of $\sigma_0^2$, the sample standard deviation $S_i$ is a biased estimator of $\sigma_0$. Its expectation is

$$
E(S_i) = c_4\sigma_0,
$$

where $c_4$ is a constant that depends on the subgroup size $n$:

$$
c_4 =
\sqrt{\frac{2}{n-1}}
\frac{\Gamma(n/2)}{\Gamma((n-1)/2)}.
$$

Since

$$
E(S_i^2)=\sigma_0^2,
$$

the variance of $S_i$ can be derived as

$$
\begin{aligned}
Var(S_i)
&= E(S_i^2)-[E(S_i)]^2 \\
&= \sigma_0^2 - (c_4\sigma_0)^2 \\
&= \sigma_0^2(1-c_4^2).
\end{aligned}
$$

Therefore, the standard deviation of $S_i$ is

$$
SD(S_i)=\sigma_0\sqrt{1-c_4^2}.
$$

The center line of the S chart is the expected value of $S_i$:

$$
CL = E(S_i) = c_4\sigma_0.
$$

Using three-sigma control limits, the upper control limit is

$$
\begin{aligned}
UCL
&= E(S_i)+3SD(S_i) \\
&= c_4\sigma_0 + 3\sigma_0\sqrt{1-c_4^2} \\
&= \left(c_4 + 3\sqrt{1-c_4^2}\right)\sigma_0.
\end{aligned}
$$

Define

$$
B_6 = c_4 + 3\sqrt{1-c_4^2}.
$$

Then,

$$
UCL = B_6\sigma_0.
$$

Similarly, the lower control limit is

$$
\begin{aligned}
LCL
&= E(S_i)-3SD(S_i) \\
&= c_4\sigma_0 - 3\sigma_0\sqrt{1-c_4^2} \\
&= \left(c_4 - 3\sqrt{1-c_4^2}\right)\sigma_0.
\end{aligned}
$$

Since the sample standard deviation cannot be negative, define

$$
B_5 = \max\left\{c_4 - 3\sqrt{1-c_4^2}, 0\right\}.
$$

Therefore,

$$
LCL = B_5\sigma_0.
$$

Hence, the S chart control limits can be written as

$$
CL = c_4\sigma_0,
$$

$$
UCL = B_6\sigma_0,
$$

$$
LCL = B_5\sigma_0.
$$

In [6]:
def simulate_ARL0_schart(c4, b5, b6,  n, mu0=0, sigma0=1, n_sim=10000, max_run=100000):
  # max_run：每次模擬最多允許抽取的 subgroup 數，避免在極端情況下長時間沒有 signal 造成無限迴圈
  # 這個函數的目的是模擬製程在正常狀態下，S chart 平均需要連續監控幾組 subgroup 才會第一次發出 signal，並以此估計 ARL0。

  # Step 1：建立一個 list，用來儲存每一次模擬得到的 run length
  # run length 表示從開始監控到第一次發出 signal，總共抽了幾組 subgroup
  run_lengths = []

  # Step 2：建立 S chart 的管制界限
  center_line = c4 * sigma0
  UCL = b6 * sigma0
  LCL = b5 * sigma0

  # Step 3：重複進行 n_sim 次模擬
  for _ in range(n_sim): 
    # Step 4：初始化本次模擬的 run length 與 signal 狀態
    run_length = 0  
    signal = False  

    # Step 5：持續抽 subgroup，直到第一次超出控制界限，或達到 max_run 為止
    while (not signal) and (run_length < max_run):
      # Step 6：每抽一組 subgroup，run length 增加 1
      run_length += 1 

      # Step 7：產生一組大小為 n 的 subgroup
      subgroup = np.random.normal(loc=mu0, scale=sigma0, size=n)

      # Step 8：計算 subgroup sample standard deviation，也就是 S chart 監控的統計量
      S = np.std(subgroup, ddof=1)

      # Step 9：判斷 S 是否超出控制界限
      if (S > UCL) or (S < LCL):
        signal = True 

    # Step 10：記錄本次模擬得到的 run length
    run_lengths.append(run_length)

  # Step 11：計算所有 run length 的平均值，作為 ARL0 的估計值
  result = {
      "ARL0": np.mean(run_lengths)
  }

  return result


In [7]:
np.random.seed(123)

# S chart 不同 subgroup size 的 c4, b5, b6
# 每一組設定包含 subgroup size n，以及對應的 S chart 常數 c4, b5, b6
settings = [
    {"n": 2, "c4": 0.7979, "b5": 0.000, "b6": 2.606},
    {"n": 3, "c4": 0.8862, "b5": 0.000, "b6": 2.276},
    {"n": 4, "c4": 0.9213, "b5": 0.000, "b6": 2.088},
    {"n": 5, "c4": 0.9400, "b5": 0.000, "b6": 1.964},
    {"n": 6, "c4": 0.9515, "b5": 0.029, "b6": 1.874}
]
schart_results = []

for setting in settings:
    n = setting["n"]
    c4 = setting["c4"]
    b5 = setting["b5"]
    b6 = setting["b6"]

    result = simulate_ARL0_schart(
        c4=c4,
        b5=b5,
        b6=b6,
        n=n
    )

    schart_results.append({
        "n": n,
        "c4": c4,
        "b5": b5,
        "b6": b6,
        "ARL0": result["ARL0"]
    })

# 轉成 DataFrame
schart_result_df = pd.DataFrame(schart_results)
print(schart_result_df)

   n      c4     b5     b6      ARL0
0  2  0.7979  0.000  2.606  107.9720
1  3  0.8862  0.000  2.276  179.4475
2  4  0.9213  0.000  2.088  224.6879
3  5  0.9400  0.000  1.964  260.5394
4  6  0.9515  0.029  1.874  284.9405
